# Step 1: Create Inverted Index

### 1.1 Parse each document in the collection to extract terms from the text fields.

I load the data from unzipped file A2 from folder documents_cs and documents_en to two dicts, indexed as {'Article_name': Article_txt}. If Article_name is not provided I name the article as " f'unnamed_article_no_{i}' ", where index i runs from 0. Then in the folowing analysis I work with these two dicts.

In [103]:
# Imports
import os
from lxml import etree
import re
#!pip install simplemma
import simplemma
from simplemma import lemmatize
from collections import defaultdict
import pandas as pd
import sacremoses
from sacremoses import MosesTokenizer


In [83]:
def load_the_xml_data_from_file_to_dict(lang: str):
    if lang not in ['CZE', 'EN']:
        raise ValueError("Invalid language. Use 'CZE' or 'EN'.")
    
    Dict_of_ARTICLES = {}
    working_dir = os.getcwd()
    parser = etree.XMLParser(recover=True)

    i = 0  # index for unnamed articles

    if lang == 'CZE':
        folder = 'documents_cs'
        file_list = os.listdir(f'{working_dir}/A2/{folder}')
        file_list.remove('czech.dtd')

        # title_tag is the unique identification ID of the document
        title_tag = 'DOCNO'
        text_tag = 'TEXT'

        for filename in file_list:
            file_path = f'{working_dir}/A2/{folder}/{filename}'
            tree = etree.parse(file_path, parser=parser)
            root = tree.getroot()

            for doc in root.iter('DOC'):
                title_elem = doc.find(title_tag)
                text_elem = doc.find(text_tag)

                title = title_elem.text.strip() if title_elem is not None and title_elem.text else f'unnamed_article_no_{i}'
                text = text_elem.text.strip() if text_elem is not None and text_elem.text else 'no_text_here'

                Dict_of_ARTICLES[title] = str(text)
                if title.startswith('unnamed_article_no_'):
                    i += 1

    elif lang == 'EN':
        folder = 'documents_en'
        file_list = os.listdir(f'{working_dir}/A2/{folder}')
        for f in ['latimes2002.dtd', 'README']:
            if f in file_list:
                file_list.remove(f)

        for filename in file_list:
            file_path = f'{working_dir}/A2/{folder}/{filename}'
            tree = etree.parse(file_path, parser=parser)
            root = tree.getroot()

            for doc in root.iter('DOC'):
                # title_elem is the unique identification ID of the document
                title_elem = doc.find('DOCNO')
                # Collect <LD> and <TE> elements
                lead_texts = [ld.text.strip() for ld in doc.findall('LD') if ld.text]
                main_texts = [te.text.strip() for te in doc.findall('TE') if te.text]
                full_text = ' '.join(lead_texts + main_texts).strip() or 'no_text_here'

                if title_elem is not None and title_elem.text:
                    title = title_elem.text.strip()
                else:
                    title = f'unnamed_article_no_{i}'
                    i += 1

                Dict_of_ARTICLES[title] = str(full_text)

    return Dict_of_ARTICLES


In [84]:
# print results, as check
CZE_Dict_of_ARTICLES = load_the_xml_data_from_file_to_dict(lang='CZE')
EN_Dict_of_ARTICLES = load_the_xml_data_from_file_to_dict(lang='EN')

### 1.2 Normalize the terms by converting them to lowercase, removing punctuation, and applying stemming or lemmatization (!).

I have decided to apply lemmatization instead of stemming, using library simplemma 1.1.2, documentation is available at: https://pypi.org/project/simplemma/

In [104]:
def normalize(lang:str):
    ''' Loads and normalizes text documents for the specified language by performing lowercasing,
        symbol removal, tokenization, and lemmatization.'''

    if lang not in ['CZE', 'EN']:
        raise ValueError("Invalid language. Use 'CZE' or 'EN'.")
    
    CZE_Dict_of_ARTICLES = load_the_xml_data_from_file_to_dict(lang=lang)
    EN_Dict_of_ARTICLES = load_the_xml_data_from_file_to_dict(lang=lang)
    
    # Set language and corresponding dictionary
    Dict_of_ARTICLES = EN_Dict_of_ARTICLES if lang == 'EN' else CZE_Dict_of_ARTICLES
    Dict_of_ARTICLES_processed = {}

    # Process articles
    for name, value in Dict_of_ARTICLES.items():
        lowercased_txt = value.lower()
        clean_text = re.sub(r'[\n\\]', ' ', lowercased_txt) # I also need to clean some symbols that I included in the text while loading in the data.
        clean_text = re.sub(r'[^\w\s]', '', clean_text)

        if lang == 'EN':
            mt = MosesTokenizer(lang='en')
            tokenized_txt = mt.tokenize(clean_text, return_str=False)
            tokens = [simplemma.lemmatize(t, lang='en') for t in tokenized_txt]

        elif lang == 'CZE':
            mt = MosesTokenizer(lang='cs')
            tokenized_txt = mt.tokenize(clean_text, return_str=False)
            tokens = [simplemma.lemmatize(t, lang='cs') for t in tokenized_txt]

        Dict_of_ARTICLES_processed[name] = tokens

    return Dict_of_ARTICLES_processed
    


In [ ]:
# print results, as check
EN_Dict_of_ARTICLES_processed = normalize(lang='EN')
CZE_Dict_of_ARTICLES_processed = normalize(lang='CZE') 

In [ ]:
# Number of all tokens (after punctuation removal and term normalisation)
print(f'Number of all tokens for CZE (after punctuation removal and term normalisation): {sum(len(tokens) for tokens in CZE_Dict_of_ARTICLES_processed.values())}')
print(f'Number of all tokens for EN (after punctuation removal and term normalisation): {sum(len(tokens) for tokens in EN_Dict_of_ARTICLES_processed.values())}') # The number makes sense, English has significantly more and longer documents
print()
# Number of all unique terms (size of the dictionary)
print(f'Number of all unique terms for CZE (size of the dictionary): {len(CZE_Dict_of_ARTICLES_processed)}')
print(f'Number of all unique terms for EN (size of the dictionary): {len(EN_Dict_of_ARTICLES_processed)}')


Number of all tokens for CZE (after punctuation removal and term normalisation): 17341285
Number of all tokens for EN (after punctuation removal and term normalisation): 47850081

Number of all unique terms for CZE (size of the dictionary): 81735
Number of all unique terms for EN (size of the dictionary): 88110


### 1.3 For each term, add the document IDs to the postings lists in lexicographical order.

In [106]:
def create_inverted_index_dict(lang: str):
    '''Creates an inverted index for a collection of documents in the specified language.'''

    if lang not in ['CZE', 'EN']:
        raise ValueError("Invalid language. Use 'CZE' or 'EN'.")
    
    EN_Dict_of_ARTICLES_processed = normalize(lang='EN')
    CZE_Dict_of_ARTICLES_processed = normalize(lang='CZE')
    
    Inverted_index = defaultdict(list)
    Dict_of_ARTICLES = EN_Dict_of_ARTICLES_processed if lang == 'EN' else CZE_Dict_of_ARTICLES_processed

    for name, value in Dict_of_ARTICLES.items():
        # Handle invalid inputs (e.g., None or non-list)
        if not isinstance(value, list) or value is None:
            value = []
        
        # Create set of unique tokens to avoid duplicates
        unique_tokens = set(value)

        # Add document name to postings list for each term
        for term in unique_tokens:
            if term:  # Skip empty terms
                Inverted_index[term].append(name)

    # Sort postings lists lexicographically once at the end
    for term in Inverted_index:
        Inverted_index[term].sort()

    return dict(Inverted_index)

In [107]:
# print results, as check
Inverted_index_CZE = create_inverted_index_dict(lang='CZE')
Inverted_index_EN = create_inverted_index_dict(lang='EN')

In [138]:
# Number of all postings (lenght of all postings lists)
length_of_all_postings_CZE = 0
length_of_all_postings_EN = 0
for key_cze, value_cze in Inverted_index_CZE.items():
    length_of_all_postings_CZE += len(value_cze)

for key_en, value_en in Inverted_index_EN.items():
    length_of_all_postings_EN += len(value_en)

print(f'Number of all postings (lenght of all postings lists) for CZE: {length_of_all_postings_CZE}')
print(f'Number of all postings (lenght of all postings lists) for EN: {length_of_all_postings_EN}')
print()
# The term with the highest document frequency (longest postings list)
key_with_longest_list_CZE = max(Inverted_index_CZE, key=lambda k: len(Inverted_index_CZE[k]))
longest_list_CZE = Inverted_index_CZE[key_with_longest_list_CZE]

key_with_longest_list_EN = max(Inverted_index_EN, key=lambda k: len(Inverted_index_EN[k]))
longest_list_EN = Inverted_index_EN[key_with_longest_list_EN]

print(f'The term with the highest document frequency (longest postings list) for CZE: {key_with_longest_list_CZE}')
print(f'The term with the highest document frequency (longest postings list) for EN: {key_with_longest_list_EN}')
print()
# The highest document frequency (length of the longest postings list)
print(f'The highest document frequency (length of the longest postings list) for CZE: {len(longest_list_CZE)}')
print(f'The highest document frequency (length of the longest postings list) for EN: {len(longest_list_EN)}')
print()
# The average document frequency (average length of the postings lists)
print(f'The average document frequency (average length of the postings lists) for CZE: {length_of_all_postings_CZE / len(Inverted_index_CZE):.1f}')
print(f'The average document frequency (average length of the postings lists) for EN: {length_of_all_postings_EN / len(Inverted_index_EN):.1f}')

Number of all postings (lenght of all postings lists) for CZE: 9188537
Number of all postings (lenght of all postings lists) for EN: 22227928

The term with the highest document frequency (longest postings list) for CZE: v
The term with the highest document frequency (longest postings list) for EN: the

The highest document frequency (length of the longest postings list) for CZE: 72714
The highest document frequency (length of the longest postings list) for EN: 85104

The average document frequency (average length of the postings lists) for CZE: 30.2
The average document frequency (average length of the postings lists) for EN: 51.1


# Step 2: Implement Boolean Query Operators

Implement Boolean query operators for two terms: x AND y, x OR y, and x AND NOT y by iterating over the sorted postings lists.

In [108]:
def boolean_and(postings_x: list, postings_y: list):
    """Returns the intersection of two sorted postings lists (x AND y)."""

    result = []
    i = 0  # Pointer for postings_x
    j = 0  # Pointer for postings_y
    
    while i < len(postings_x) and j < len(postings_y):
        if postings_x[i] == postings_y[j]:
            result.append(postings_x[i])
            i += 1
            j += 1
        elif postings_x[i] < postings_y[j]:
            i += 1
        else:
            j += 1
            
    return result

def boolean_or(postings_x: list, postings_y: list):
    """Returns the union of two sorted postings lists (x OR y)."""

    result = []
    i = 0  # Pointer for postings_x
    j = 0  # Pointer for postings_y
    
    while i < len(postings_x) and j < len(postings_y):
        if postings_x[i] == postings_y[j]:
            result.append(postings_x[i])
            i += 1
            j += 1
        elif postings_x[i] < postings_y[j]:
            result.append(postings_x[i])
            i += 1
        else:
            result.append(postings_y[j])
            j += 1
    
    # Append remaining elements
    result.extend(postings_x[i:])
    result.extend(postings_y[j:])
    
    return result

def boolean_and_not(postings_x: list, postings_y: list):
    """Returns documents in postings_x but not in postings_y (x AND NOT y)."""

    result = []
    i = 0  # Pointer for postings_x
    j = 0  # Pointer for postings_y
    
    while i < len(postings_x):
        if j >= len(postings_y):  # No more y documents to exclude
            result.append(postings_x[i])
            i += 1
        elif postings_x[i] == postings_y[j]:
            i += 1  # Skip documents in both lists
            j += 1
        elif postings_x[i] < postings_y[j]:
            result.append(postings_x[i])  # x document not in y
            i += 1
        else:
            j += 1  # Skip y document
            
    return result

In [109]:
def process_boolean_query(term_x: str, term_y: str, inverted_index: dict):
    """Processes a Boolean query for two terms and returns results for AND, OR, AND NOT."""
    
    # Get postings lists, default to empty list if term not found
    postings_x = inverted_index.get(term_x, [])
    postings_y = inverted_index.get(term_y, [])
    
    # Compute results for each operator
    return {
        'AND': boolean_and(postings_x, postings_y),
        'OR': boolean_or(postings_x, postings_y),
        'AND NOT': boolean_and_not(postings_x, postings_y)
    }

# Step 3: Process Boolean Queries

Process the Boolean queries for the provided topics separately for Czech and English and generate the results.

### 3.1 Parse the topic file to extract queries.

In [110]:
working_dir = os.getcwd()
parser = etree.XMLParser(recover=True)

# Parse Czech document
file_path_CZE = f'{working_dir}/A2/topics-train_cs.xml'
file_path_EN = f'{working_dir}/A2/topics-train_en.xml'

def parse_queries_lxml(xml_file_path):
    tree = etree.parse(xml_file_path)
    root = tree.getroot()

    queries = []
    for top in root.findall('top'):
        num = top.findtext('num').strip()
        lang = top.get('lang', 'unknown')
        title = top.findtext('title').strip()
        query = top.findtext('query').strip()

        queries.append({
            'id': num,
            'lang': lang,
            'title': title,
            'query': query
        })

    return queries
    

In [111]:
# print results, as check
CZE_querries = parse_queries_lxml(file_path_CZE)
EN_querries = parse_queries_lxml(file_path_EN)

### 3.2 Tokenize and normalize the terms in the queries the same way as for the document collection.

Hence I apply on the querries the same transformation as on the data above.

In [112]:
def normalize_query(querry:str, lang: str):
    # Process querry
    lowercased_querry = querry.lower()
    clean_querry = re.sub(r'[\n\\]', ' ', lowercased_querry)
    clean_querry = re.sub(r'[^\w\s]', '', clean_querry)

    tokenized_clean_querry = clean_querry.split()
    if lang == 'EN':
        tokens = [simplemma.lemmatize(t, lang='en') for t in tokenized_clean_querry]
    elif lang == 'CZE':
        tokens = [simplemma.lemmatize(t, lang='cs') for t in tokenized_clean_querry]


    return tokens

In [113]:
CZE_querries_normalized = []
for cze_querry in CZE_querries:
    querry = normalize_query(cze_querry['query'], 'CZE')

    if 'AND NOT' in cze_querry['query']:
        logic_operator = 'AND NOT'

    elif 'AND' in cze_querry['query']:
        logic_operator = 'AND'

    elif 'OR' in cze_querry['query']:
        logic_operator = 'OR'

    else:
        logic_operator = None

    CZE_querries_normalized.append((querry, logic_operator, cze_querry['id']))


EN_querries_normalized = []
for en_querry in EN_querries:
    querry = normalize_query(en_querry['query'], 'EN')

    if 'AND NOT' in en_querry['query']:
        logic_operator = 'AND NOT'
    
    elif 'AND' in en_querry['query']:
        logic_operator = 'AND'

    elif 'OR' in en_querry['query']:
        logic_operator = 'OR'

    else:
        logic_operator = None

    EN_querries_normalized.append((querry, logic_operator, en_querry['id']))
    
    

### 3.3 Use the inverted index to retrieve document IDs that match each query using the implemented operators.


In [114]:
Querry_results_CZE = []

for querry, logic_operator,querry_id in CZE_querries_normalized:
    if logic_operator == 'AND NOT':
        output_key = 'AND NOT'

    elif logic_operator == 'OR':
        output_key = 'OR'

    elif logic_operator == 'AND':
        output_key = 'AND'

    # querry: ['word to look for 1', 'boolean operation which we include elsewhere', 'word to look for 2', hence we input querry[0], querry[2]
    Querry_results_CZE.append(((process_boolean_query(querry[0], querry[2], Inverted_index_CZE)[output_key]), querry_id)
                              )
    

In [115]:
Querry_results_EN = []

for querry, logic_operator, querry_id in EN_querries_normalized:
    if logic_operator == 'AND NOT':
        output_key = 'AND NOT'

    elif logic_operator == 'OR':
        output_key = 'OR'

    elif logic_operator == 'AND':
        output_key = 'AND'

    # querry: ['word to look for 1', 'boolean operation which we include elsewhere', 'word to look for 2', hence we input querry[0], querry[2]
    Querry_results_EN.append(((process_boolean_query(querry[0], querry[2], Inverted_index_EN)[output_key]), querry_id)
                             )
    

### 3.4 Save the results for each query in the required format.
### 3.5 Ensure the results are saved in files named results-cs.dat and results-en.dat for evaluation for the Czech and English results respectively.

In [116]:
def save_query_results_to_file(query_results: list, filename: str = 'results-cs.dat'):
    """
    Saves a list of query results to a file in the format: <query_id> 0 <doc_id> 0.
    
    Args:
        query_results: List of tuples, each containing (list of doc_ids, query_id).
        filename: Output file name (default: 'results-cs.dat').
    """
    with open(filename, 'w', encoding='utf-8') as f:
        for doc_ids, query_id in query_results:
            for doc_id in doc_ids:
                f.write(f"{query_id} 0 {doc_id} 0\n")


In [117]:
save_query_results_to_file(Querry_results_CZE, filename= 'results-cs.dat')
save_query_results_to_file(Querry_results_EN, filename= 'results-en.dat')

# Step 4: Evaluate Results

### 4.1 For each query, compute the number of retrieved documents, the number of relevant documents retrieved, Precision, and Recall.
### 4.2 Average the values of all the scores across all queries to obtain the overall scores.

In [146]:
# DataFrames from Querry_results
print('for CZE:')
cze_data = [{'query_id': query_id, 'retrieved_docs_count': len(docs)} for docs, query_id in Querry_results_CZE]
df_cze = pd.DataFrame(cze_data)
print(df_cze)

print('for EN:')
en_data = [{'query_id': query_id, 'retrieved_docs_count': len(docs)} for docs, query_id in Querry_results_EN]
df_en = pd.DataFrame(en_data)
print(df_en)

print('\n')
total_retrieved_docs_CZE = df_cze['retrieved_docs_count'].sum()
total_retrieved_docs_EN = df_en['retrieved_docs_count'].sum()
print(f"Total retrieved documents for CZE: {total_retrieved_docs_CZE}")
print(f"Total retrieved documents for EN: {total_retrieved_docs_EN}")

# Load files
working_dir = os.getcwd()
results_cs_path = os.path.join(working_dir, 'results-cs.dat')
qrels_cs_path = os.path.join(working_dir, 'A2', 'qrels-train_cs.txt')
results_en_path = os.path.join(working_dir, 'results-en.dat')
qrels_en_path = os.path.join(working_dir, 'A2', 'qrels-train_en.txt')

for path in [results_cs_path, qrels_cs_path, results_en_path, qrels_en_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing: {path}")

df_cs_dat = pd.read_csv(results_cs_path, sep=' ', header=None, names=['qid', '1 (ignore)', 'docno', '2 (ignore)'])
df_en_dat = pd.read_csv(results_en_path, sep=' ', header=None, names=['qid', '1 (ignore)', 'docno', '2 (ignore)'])
df_cs_test = pd.read_csv(qrels_cs_path, sep=' ', header=None, names=['qid', 'iter (ignore)', 'docno', 'rel'])
df_en_test = pd.read_csv(qrels_en_path, sep=' ', header=None, names=['qid', 'iter (ignore)', 'docno', 'rel'])

# Filter relevant documents
df_cs_test_relevant = df_cs_test[df_cs_test['rel'] >= 1]
df_en_test_relevant = df_en_test[df_en_test['rel'] >= 1]

# Unique query IDs
unique_qids_CZE = df_cs_dat['qid'].unique()
unique_qids_EN = df_en_dat['qid'].unique()

# Metrics
precisions_CZE, recalls_CZE = [], []
precisions_EN, recalls_EN = [], []
tp_total_CZE, relevant_total_CZE = 0, 0
tp_total_EN, relevant_total_EN = 0, 0

# CZE
for qid_cze in unique_qids_CZE:
    retrieved_cze = df_cs_dat[df_cs_dat['qid'] == qid_cze]
    relevant_cze = df_cs_test_relevant[df_cs_test_relevant['qid'] == qid_cze]
    retrieved_docs_cze = set(retrieved_cze['docno'])
    relevant_docs_cze = set(relevant_cze['docno'])
    tp_cze = len(retrieved_docs_cze & relevant_docs_cze)
    
    precision_cze = tp_cze / len(retrieved_cze) if len(retrieved_cze) > 0 else 0
    recall_cze = tp_cze / len(relevant_cze) if len(relevant_cze) > 0 else 0
    precisions_CZE.append(precision_cze)
    recalls_CZE.append(recall_cze)
    tp_total_CZE += tp_cze
    relevant_total_CZE += len(relevant_cze)
    
    print(f"Query {qid_cze} (CZE): Precision = {precision_cze:.4f}, Recall = {recall_cze:.4f}")
    print(f"Query {qid_cze} (CZE): Relevant retrieved: {tp_cze}/{len(retrieved_cze)}")

# EN
for qid_en in unique_qids_EN:
    retrieved_en = df_en_dat[df_en_dat['qid'] == qid_en]
    relevant_en = df_en_test_relevant[df_en_test_relevant['qid'] == qid_en]
    retrieved_docs_en = set(retrieved_en['docno'])
    relevant_docs_en = set(relevant_en['docno'])
    tp_en = len(retrieved_docs_en & relevant_docs_en)
    
    precision_en = tp_en / len(retrieved_en) if len(retrieved_en) > 0 else 0
    recall_en = tp_en / len(relevant_en) if len(relevant_en) > 0 else 0
    precisions_EN.append(precision_en)
    recalls_EN.append(recall_en)
    tp_total_EN += tp_en
    relevant_total_EN += len(relevant_en)
    
    print(f"Query {qid_en} (EN): Precision = {precision_en:.4f}, Recall = {recall_en:.4f}")
    print(f"Query {qid_en} (EN): Relevant retrieved: {tp_en}/{len(retrieved_en)}")

# Aggregate Stats
print("==== Aggregate Statistics ====")
print(f"Average Precision (CZE): {sum(precisions_CZE) / len(precisions_CZE):.4f}")
print(f"Average Recall    (CZE): {sum(recalls_CZE) / len(recalls_CZE):.4f}")
print(f"Average Precision (EN):  {sum(precisions_EN) / len(precisions_EN):.4f}")
print(f"Average Recall    (EN):  {sum(recalls_EN) / len(recalls_EN):.4f}")
print(f"In total for CZE, {tp_total_CZE} relevant documents retrieved out of {relevant_total_CZE} relevant, relative to {len(df_cs_dat)} total retrieved.")
print(f"In total for EN, {tp_total_EN} relevant documents retrieved out of {relevant_total_EN} relevant, relative to {len(df_en_dat)} total retrieved.")

for CZE:
          query_id  retrieved_docs_count
0   10.2452/401-AH                    44
1   10.2452/402-AH                    19
2   10.2452/403-AH                    32
3   10.2452/404-AH                   336
4   10.2452/405-AH                     5
5   10.2452/406-AH                    85
6   10.2452/407-AH                    11
7   10.2452/408-AH                    14
8   10.2452/409-AH                     6
9   10.2452/410-AH                    25
10  10.2452/411-AH                     2
11  10.2452/412-AH                    84
12  10.2452/413-AH                     0
13  10.2452/414-AH                     9
14  10.2452/415-AH                   784
15  10.2452/416-AH                    22
16  10.2452/417-AH                    20
17  10.2452/418-AH                     0
18  10.2452/419-AH                    17
19  10.2452/420-AH                     8
20  10.2452/421-AH                     0
21  10.2452/422-AH                    86
22  10.2452/423-AH                   119
23  10.